# GymLeader: GRPO of the SFT model on Google Colab

Reinforcement learning (GRPO) on top of the SFT checkpoint: for each training request the model writes 8 answers, the
project's verifier scores them (`verify()["reward"]`), and the model is pushed toward the answers that scored above their
group. It follows Unsloth's Qwen3.5 guide (https://unsloth.ai/docs/models/qwen3.5/fine-tune): no vLLM for Qwen3.5, so
`fast_inference=False`. The result goes to the Hugging Face Hub, and the Mac evaluates it from there.

## How to run
1. Runtime → Change runtime type → **T4 GPU** or better (L4, A100).
2. Add the Colab secret `HF_TOKEN` (key icon on the left): a token with **write** access from
   https://huggingface.co/settings/tokens, with **notebook access** switched on. Your username is read from the token.
3. Run the install cell, then **Runtime → Restart session**.
4. Click the second cell and choose **Runtime → Run cell and below**. Each section asserts its own result, so a problem
   stops the run where it happens.

The adapter is uploaded every 50 steps and again right after training: a crash or a disconnect never means training from
scratch (see section 2). `test.jsonl` is never loaded: the test set is only for the final evaluation on the Mac.

## 1. Install

The same install as the SFT notebook: Qwen3.5 needs transformers v5 and Colab ships v4, so we follow Unsloth's forced,
no-cache reinstall and ask for v5 in the same command. Restart the session afterwards.

**Check:** pip ends without errors.

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo "transformers>=5,<6"

## 2. Versions, settings, GPU and Hugging Face login

Everything that can fail does so here, before the slow steps: the transformers version, the GPU, the Hugging Face token
(write access; `HF_USER`, the owner of the new repos, is read from it) and the two recovery settings below.
All settings live in the second cell: each step uses 2 prompts × 8 answers (1 × 8 if the padding check in section 7 fails),
and 300 steps × 2 prompts use every GRPO prompt once.
Training stops at `MAX_STEPS` or when `TIME_BUDGET_MIN` is used up, whichever comes first.

**`START_ADAPTER` / `SKIP_TRAINING`:** keep the defaults (start from the SFT adapter, train) for a normal run.
After a crash *after* training (curves, merge, upload…): `START_ADAPTER = "<HF_USER>/qwen3.5-2b-gym-grpo-lora"` and `SKIP_TRAINING = True` skip sections 7–9 and redo the rest on the uploaded final adapter.
After a disconnect *during* training: the same `START_ADAPTER` with `SKIP_TRAINING = False` restarts training from the last 50-step checkpoint; the optimizer state is lost, which is acceptable.
A resumed run starts again from the first prompt, and its step-0 check measures the checkpoint, not SFT.

**Check:** transformers 5.x; the run line with the settings; the GPU name and memory; "logged in to Hugging Face as <HF_USER>".

In [ ]:
import unsloth  # import first: Unsloth patches transformers/trl when it is imported
import math, os, time, torch, transformers, trl
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi

for lib in (unsloth, transformers, trl, torch):
    print(f"{lib.__name__:13} {lib.__version__}")
assert transformers.__version__.startswith("5."), "Qwen3.5 needs transformers v5: restart the session and re-run"

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-2B"
SFT_ADAPTER = "riccardoroselli/qwen3.5-2b-gym-sft-lora"
START_ADAPTER = SFT_ADAPTER    # the uploaded GRPO adapter only to recover a run (see above)
SKIP_TRAINING = False          # True: skip sections 7-9 and use START_ADAPTER as the trained adapter
MAX_SEQ_LENGTH = 1024
TEMPERATURE = 0.7              # sampling temperature of the G answers (0.7 vs 1.0 compared with src/probe_groups.py)
G = 8                          # answers per prompt (num_generations)
PROMPTS_PER_STEP = 2           # prompts per optimizer step (16 answers); section 7 sets 1 if the padding check fails
MAX_COMPLETION = 200           # new tokens per answer: the probe's longest well-formed answer was 152
BETA = 0.0                     # no KL penalty
LEARNING_RATE = 5e-6           # Unsloth's GRPO notebook
WARMUP_STEPS = 10              # then constant
MAX_STEPS = 300
TIME_BUDGET_MIN = 100          # stop training after this many minutes, whatever the step
UPLOAD_EVERY = VAL_EVERY = 50  # steps
SEED = 42
RUN_NAME = "sft_grpo"
RUN_DIR = Path("/content/run") / RUN_NAME
N_PROMPTS = MAX_STEPS * PROMPTS_PER_STEP  # 600: with 2 prompts per step, every GRPO prompt is used once
PROMPT_MIX = {"impossible": 0.20, 3: 0.15, 4: 0.20, 5: 0.225, 6: 0.225}  # numbers = feasible with n exercises

assert not (SKIP_TRAINING and START_ADAPTER.endswith("/qwen3.5-2b-gym-sft-lora")), \
    "SKIP_TRAINING=True needs the trained GRPO adapter as START_ADAPTER, not the SFT one"
assert abs(sum(PROMPT_MIX.values()) - 1) < 1e-9, "PROMPT_MIX shares must add up to 1"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"run {RUN_NAME} -> {RUN_DIR}: up to {MAX_STEPS} steps x {PROMPTS_PER_STEP} prompts x {G} answers, "
      f"{TIME_BUDGET_MIN} min budget; start {START_ADAPTER}; skip training: {SKIP_TRAINING}")

In [ ]:
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.is_bf16_supported()
print(f"GPU {gpu.name}, {gpu.total_memory / 1e9:.1f} GB, bf16 {BF16}")
if "T4" in gpu.name:
    print("Note: T4 detected. Qwen3.5's kernels compile slowly on T4, so the first steps will be slow. That is expected.")

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # fails if the secret is missing or notebook access is off
api = HfApi()  # the token above also lets from_pretrained() download the private GRPO adapter (recovery runs)
me = api.whoami()
HF_USER = me["name"]  # the new repos are created under this account
role = me.get("auth", {}).get("accessToken", {}).get("role")
assert role != "read", "HF_TOKEN is read-only: the uploads need a token with write access"
print(f"logged in to Hugging Face as {HF_USER} (token role: {role})")
GRPO_ADAPTER_REPO, GRPO_MERGED_REPO = f"{HF_USER}/qwen3.5-2b-gym-grpo-lora", f"{HF_USER}/qwen3.5-2b-gym-grpo"

if SKIP_TRAINING:  # only section 9's final upload makes an adapter safe to use without training
    last = api.list_repo_commits(START_ADAPTER)[0].title
    assert "final" in last, f"the latest commit of {START_ADAPTER} is {last!r}, not a final upload: resume training instead"
    print(f"SKIP_TRAINING: using {START_ADAPTER} ({last!r})")

## 3. Data

We clone `main` for today's code, but the data must be the frozen dataset tagged `data-v1`, so we assert that
`data/processed/` is identical to the tag. Only train and val are loaded. The GRPO prompts are 600 train rows drawn without
replacement with a fixed seed, in the mix of section 2: 20% impossible requests (6% in train), because refusing is what
SFT learned least.

**Check:** "data/processed/ matches data-v1", "train 2000, val 200", and every share in the mix table at its target.

In [ ]:
import collections, json, random, subprocess, sys

GITHUB = "https://github.com/riccardoroselli/SFT-Gym-Workout-Program"
REPO = Path("/content/SFT-Gym-Workout-Program")
if not REPO.exists():
    !git clone --quiet --branch main {GITHUB}.git {REPO}
!git -C {REPO} fetch --quiet --tags
!git -C {REPO} log -1 --oneline
diff = subprocess.run(["git", "-C", str(REPO), "diff", "--quiet", "data-v1", "HEAD", "--", "data/processed/"])
assert diff.returncode == 0, "data/processed/ on main differs from data-v1: the training data is not the frozen dataset"
CODE_COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                             capture_output=True, text=True, check=True).stdout.strip()
print(f"data/processed/ matches data-v1; code commit {CODE_COMMIT[:7]}")
sys.path.insert(0, str(REPO / "src"))  # so we can import evaluate.py and verifier.py

from evaluate import read_jsonl
train = read_jsonl(REPO / "data/processed/train.jsonl")
val = read_jsonl(REPO / "data/processed/val.jsonl")
assert (len(train), len(val)) == (2000, 200), (len(train), len(val))
print(f"train {len(train)}, val {len(val)}")

In [ ]:
def category(meta):
    return "impossible" if not meta["feasible"] else meta["n_exercises"]


def grpo_rows(rows, n_total, mix, seed):
    "n_total rows drawn without replacement: round(share * n_total) per category of `mix`, then shuffled."
    rng = random.Random(seed)
    counts = {k: round(share * n_total) for k, share in mix.items()}
    assert sum(counts.values()) == n_total, f"the mix doesn't round to {n_total} prompts: {counts}"
    picked = []
    for k, n in counts.items():
        pool = [r for r in rows if category(r["meta"]) == k]
        assert n <= len(pool), f"{k}: {n} prompts needed, only {len(pool)} train rows"
        picked += rng.sample(pool, n)
    rng.shuffle(picked)
    return picked


grpo_train = grpo_rows(train, N_PROMPTS, PROMPT_MIX, SEED)
real = collections.Counter(category(r["meta"]) for r in grpo_train)
print("| prompts | count | share | target |\n|---|---|---|---|")
for k, share in PROMPT_MIX.items():
    print(f"| {k if k == 'impossible' else f'feasible n={k}'} | {real[k]} | {real[k] / N_PROMPTS:.4f} | {share} |")
    assert abs(real[k] / N_PROMPTS - share) <= 0.5 / N_PROMPTS + 1e-9, f"{k}: share off by more than rounding"
assert len(grpo_train) == N_PROMPTS == len({id(r) for r in grpo_train}), "a train row was drawn twice"

## 4. Model

We start from the SFT adapter, not a new LoRA: GRPO keeps training the same LoRA matrices from where SFT left them.
Unsloth loads the adapter's base model in 16 bit (as in SFT), then the adapter as trainable; `fast_inference=False`
because vLLM doesn't support Qwen3.5, so GRPO generates with Unsloth's own (slower) inference. The end-of-turn token must
be `<|im_end|>`, so the model keeps stopping after the JSON.

**Check:** "10,911,744 LoRA parameters, all trainable" (the SFT count), nothing else trainable; eos_token `<|im_end|>`.

In [ ]:
from unsloth import FastLanguageModel

SFT_LORA_PARAMS = 10_911_744  # trainable parameters in the SFT run (notebook 02, section 8)
model, tokenizer = FastLanguageModel.from_pretrained(
    START_ADAPTER,  # an adapter repo: Unsloth loads its base model, then the adapter with is_trainable=True
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
    fast_inference=False,  # no vLLM for Qwen3.5
    use_gradient_checkpointing="unsloth",
)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved() / 1e9:.2f} GB reserved")
by_dtype = collections.Counter()
for p in model.parameters():
    by_dtype[str(p.dtype)] += p.numel()
for dtype, n in by_dtype.most_common():
    print(f"  {dtype:15} {n:>14,} parameters")

lora = [p for n, p in model.named_parameters() if "lora_" in n]
others = [n for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n]
n_lora = sum(p.numel() for p in lora)
print(f"base {model.peft_config['default'].base_model_name_or_path}: {n_lora:,} LoRA parameters, "
      f"all trainable: {all(p.requires_grad for p in lora)}, other trainable: {len(others)}")
assert n_lora == SFT_LORA_PARAMS and all(p.requires_grad for p in lora), "the adapter's LoRA weights are missing or frozen"
assert not others, f"non-LoRA weights are trainable: {others[:3]}"

END = "<|im_end|>"
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)  # Qwen3.5 is a vision-language model: processor -> text tokenizer
print("eos_token:", repr(text_tokenizer.eos_token), "id", text_tokenizer.eos_token_id)
assert text_tokenizer.eos_token == END

## 5. Chat template check

GRPO must sample answers to exactly the prompt the model sees at evaluation, so every GRPO prompt is built with
`build_prompt()` from `src/evaluate.py` and compared with the one the reference tokenizer builds, as in the SFT notebook.
TRL tokenizes plain-text prompts with the tokenizer's defaults, so we also check that this adds no special tokens.

**Check:** "template check passed on 600 prompts", and prompt + answer fit in `MAX_SEQ_LENGTH`.

In [ ]:
from transformers import AutoTokenizer
from evaluate import build_prompt

reference = AutoTokenizer.from_pretrained(MODEL_NAME)
prompts = [build_prompt(text_tokenizer, r["messages"][:2]) for r in grpo_train]
print(prompts[0])
assert all(p == build_prompt(reference, r["messages"][:2]) for p, r in zip(prompts, grpo_train)), \
    "a GRPO prompt differs from the evaluate.py prompt"
ids = [text_tokenizer(text=p)["input_ids"] for p in prompts]  # how TRL tokenizes a plain-text prompt
assert all(i == text_tokenizer(p, add_special_tokens=False)["input_ids"] for i, p in zip(ids, prompts)), \
    "TRL's tokenization would add special tokens to the prompt"
longest = max(map(len, ids))
assert longest + MAX_COMPLETION <= MAX_SEQ_LENGTH
print(f"template check passed on {len(prompts)} prompts; longest {longest} tokens + {MAX_COMPLETION} new <= {MAX_SEQ_LENGTH}")

## 6. Reward

The reward is exactly `verify(request, answer)["reward"]` from `src/verifier.py`, with its default options (variant pairs
like `x` + `x v. 2` are duplicates): the same function that scores the test set, so GRPO optimizes the number we report.
It also logs, with TRL's `log_metric`, the mean of every check plus the duplicate, refusal and false-refusal rates, for the
curves in section 10. We test it on known answers before training.

**Check:** "reward check passed": 1.0 for a train reference answer, less for each mistake, and the same as `verify()`.

In [ ]:
from evaluate import FIELDS, MAX_NEW_TOKENS, mean
from verifier import verify

REWARD_FIELDS = [k for k in FIELDS if k not in ("has_think", "truncated")]


def reward_stats(metas, results):
    "Means of verify()'s checks, plus duplicate, refusal (impossible) and false-refusal (feasible) rates."
    refused = [r["parse_ok"] and r["names_exist"] is None for r in results]  # parses, but is not a workout
    stats = {k: mean(r[k] for r in results) for k in REWARD_FIELDS}
    stats["duplicate_rate"] = mean(not r["no_duplicates"] for r in results if r["no_duplicates"] is not None)
    stats["refusal_rate"] = mean(x for x, m in zip(refused, metas) if not m["feasible"])
    stats["false_refusal_rate"] = mean(x for x, m in zip(refused, metas) if m["feasible"])
    return stats


def gym_reward(prompts, completions, meta, log_metric=None, **kwargs):
    "GRPO reward: exactly verify(request, completion)['reward'], default options."
    assert log_metric is not None, "this TRL version doesn't pass log_metric to reward functions"
    results = [verify(m, c) for m, c in zip(meta, completions)]
    for name, value in reward_stats(meta, results).items():
        if value is not None:  # None: no answer of that kind in this batch
            log_metric(f"gym/{name}", value)
    return [r["reward"] for r in results]

In [ ]:
logged = collections.defaultdict(list)
log = lambda name, value: logged[name].append(value)

ref = next(r for r in train if r["meta"]["feasible"] and verify(r["meta"], r["messages"][2]["content"])["all_correct"])
meta, good = ref["meta"], ref["messages"][2]["content"]
items = json.loads(good)["workout"]
dump = lambda items: json.dumps({"workout": items}, separators=(",", ":"))
cases = {
    "reference answer": good,
    "duplicate": dump(items[:-1] + items[:1]),
    "invented name": dump(items[:-1] + [{**items[-1], "name": "super chest blaster 3000"}]),
    "wrong refusal": json.dumps({"error": "Not enough exercises."}),
    "broken JSON": good[:-3],
}
rewards = gym_reward(list(cases), list(cases.values()), [meta] * len(cases), log_metric=log)
for (name, text), reward in zip(cases.items(), rewards):
    print(f"{name:17} reward {reward:.3f}")
    assert reward == verify(meta, text)["reward"], name
assert rewards[0] == 1.0 and all(r < 1.0 for r in rewards[1:])

impossible = next(r for r in train if not r["meta"]["feasible"])
assert gym_reward([""], [impossible["messages"][2]["content"]], [impossible["meta"]], log_metric=log) == [1.0]
assert {"gym/all_correct", "gym/duplicate_rate", "gym/refusal_rate", "gym/false_refusal_rate"} <= set(logged)
print("reward check passed; logged:", ", ".join(sorted(logged)))

## 7. Val check

During training we want a number we trust more than the training reward: 40 fixed val rows (all 13 impossible ones and
27 feasible), answered greedily like `evaluate.py` and scored with `verify()`. It runs at step 0 (the starting model: the
SFT reference) and every 50 steps. Answers are generated 8 at a time, padded on the left, to keep it fast. A padding check
comes first: 6 rows of different prompt lengths, answered in one batch and one at a time. If padding changes the answers, the
val check goes one row at a time and GRPO uses 1 prompt × 8 answers per step (8 copies of one prompt need no padding).

**Check:** "40 val rows, 13 impossible"; 6 × MATCH (a DIFF with about the same reward is fine) and "padding check passed",
or the WARNING; step-0 means close to the SFT model's.

In [ ]:
VAL_BATCH = 8
IM_END_ID = text_tokenizer.convert_tokens_to_ids(END)
impossible_val = [r for r in val if not r["meta"]["feasible"]]
VAL_SET = impossible_val + random.Random(SEED).sample([r for r in val if r["meta"]["feasible"]], 40 - len(impossible_val))
assert (len(VAL_SET), len(impossible_val)) == (40, 13), (len(VAL_SET), len(impossible_val))
print(f"{len(VAL_SET)} val rows, {len(impossible_val)} impossible")
VAL_HISTORY = []  # one entry per val check: {"step", "means", "answers"}
BATCHED_GENERATION, PADDING_CHECK = True, None  # set by the padding check below


def greedy_answers(model, rows, batch_size=None):
    "Greedy answers like evaluate.py (build_prompt, max 512 new tokens, stop at <|im_end|>). -> [(text, truncated)]"
    batch_size = batch_size or (VAL_BATCH if BATCHED_GENERATION else 1)
    side, text_tokenizer.padding_side = text_tokenizer.padding_side, "left"  # generation pads on the left
    out = []
    try:
        for i in range(0, len(rows), batch_size):
            prompts_ = [build_prompt(text_tokenizer, r["messages"][:2]) for r in rows[i:i + batch_size]]
            inputs = text_tokenizer(prompts_, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                gen = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     eos_token_id=IM_END_ID, pad_token_id=text_tokenizer.pad_token_id)
            for new in gen[:, inputs["input_ids"].shape[1]:]:
                out.append((text_tokenizer.decode(new, skip_special_tokens=True), IM_END_ID not in new.tolist()))
    finally:
        text_tokenizer.padding_side = side
    return out


def val_check(model, step):
    "The 40 VAL_SET rows, greedy, scored with verify(). Leaves the model in inference mode."
    FastLanguageModel.for_inference(model)
    answers = greedy_answers(model, VAL_SET)
    results = [verify(r["meta"], text) for r, (text, _) in zip(VAL_SET, answers)]
    means = reward_stats([r["meta"] for r in VAL_SET], results)
    means["truncated"] = mean(t for _, t in answers)
    return {"step": step, "means": means,
            "answers": [{"output": text, "truncated": t, **res} for (text, t), res in zip(answers, results)]}


def save_val_history():
    (RUN_DIR / "val_checks.json").write_text(json.dumps({"padding_check": PADDING_CHECK, "checks": VAL_HISTORY}, indent=1))


def means_table(columns):
    "{label: means} -> markdown table, one row per check."
    labels, keys = list(columns), list(next(iter(columns.values())))
    lines = ["| check | " + " | ".join(labels) + " |", "|---|" + "---|" * len(labels)]
    lines += [f"| {k} | " + " | ".join("-" if columns[l][k] is None else str(columns[l][k]) for l in labels) + " |"
              for k in keys]
    return "\n".join(lines)

In [ ]:
if SKIP_TRAINING:
    print("SKIP_TRAINING: no padding check (section 9 loads the training run's result)")
else:
    FastLanguageModel.for_inference(model)
    by_length = {}
    for r in val:  # one val row per prompt length
        n = len(text_tokenizer(build_prompt(text_tokenizer, r["messages"][:2]), add_special_tokens=False)["input_ids"])
        by_length.setdefault(n, r)
    lengths = sorted(by_length)
    assert len(lengths) >= 6, lengths
    pad_rows = [by_length[lengths[round(i * (len(lengths) - 1) / 5)]] for i in range(6)]  # shortest to longest
    batched = greedy_answers(model, pad_rows, batch_size=len(pad_rows))  # the shorter prompts get left padding
    single = greedy_answers(model, pad_rows, batch_size=1)               # no padding
    pad_results = []
    for row, (b, _), (s, _) in zip(pad_rows, batched, single):
        rb, rs = verify(row["meta"], b), verify(row["meta"], s)
        ok = bool(rb["parse_ok"]) and abs(rb["reward"] - rs["reward"]) <= 0.1
        n = len(text_tokenizer(build_prompt(text_tokenizer, row["messages"][:2]), add_special_tokens=False)["input_ids"])
        pad_results.append({"prompt_tokens": n, "match": b == s, "ok": ok,
                            "reward_batched": rb["reward"], "reward_single": rs["reward"], "batched": b, "single": s})
        print("MATCH" if b == s else "DIFF ", f"{n:3} prompt tokens: reward batched {rb['reward']:.2f}, single {rs['reward']:.2f}",
              "" if ok else "<- FAIL")
    BATCHED_GENERATION = all(r["ok"] for r in pad_results)
    if BATCHED_GENERATION:
        print("padding check passed: val checks in batches of 8, GRPO with 2 prompts per step")
    else:
        PROMPTS_PER_STEP = 1
        print("WARNING: left padding changes the answers. Val checks now generate one row at a time (slower), and GRPO "
              "uses 1 prompt x 8 answers per step, so its generation batch (8 copies of one prompt) has no padding.")
    PADDING_CHECK = {"passed": BATCHED_GENERATION, "prompts_per_step": PROMPTS_PER_STEP, "rows": pad_results}
    save_val_history()

In [ ]:
if SKIP_TRAINING:
    print("SKIP_TRAINING: no step-0 val check (section 9 loads the one from the training run)")
else:
    VAL_HISTORY.append(val_check(model, 0))
    save_val_history()
    print(f"step 0 ({'batches of 8' if BATCHED_GENERATION else 'one row at a time'}):")
    print(means_table({"step 0": VAL_HISTORY[0]["means"]}))
    FastLanguageModel.for_training(model)

## 8. Trainer

TRL's `GRPOTrainer` samples G = 8 answers per prompt at `TEMPERATURE`, scores them with `gym_reward`, and pushes the model
toward the answers that scored above their group's mean. Learning rate and optimizer follow Unsloth's GRPO notebook, but the
rate stays constant after a 10-step warmup, so stopping early on the time budget doesn't cut a schedule short.
Callbacks: the time budget, the val check every 50 steps, and an adapter upload every 50 steps (a disconnect loses at most 50).

**Check:** the batch line (8 answers × 2 = 2 prompts × 8 answers per step, or × 1 = 1 prompt if the padding check failed),
the real dtypes (float32 LoRA on a T4).

In [ ]:
from transformers import TrainerCallback


class TimeBudget(TrainerCallback):
    "Stops training at the end of the first step that finishes after `minutes`."
    def __init__(self, minutes, clock=time.time):
        self.seconds, self.clock, self.start = minutes * 60, clock, None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start = self.clock()

    def on_step_end(self, args, state, control, **kwargs):
        if self.clock() - self.start >= self.seconds:
            print(f"time budget ({self.seconds / 60:.0f} min) used up at step {state.global_step}: stopping")
            control.should_training_stop = True


class ValCheck(TrainerCallback):
    "Every `every` steps: the val check of section 7, added to VAL_HISTORY and saved."
    def __init__(self, every):
        self.every = every

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.every == 0:
            VAL_HISTORY.append(val_check(model, state.global_step))
            FastLanguageModel.for_training(model)
            save_val_history()
            m = VAL_HISTORY[-1]["means"]
            print(f"step {state.global_step} val check: reward {m['reward']}, all_correct {m['all_correct']}, "
                  f"duplicates {m['duplicate_rate']}, refusals {m['refusal_rate']}")


class HubCheckpoint(TrainerCallback):
    "Every `every` steps: save the adapter and upload it, so a disconnect loses at most `every` steps."
    def __init__(self, api, repo, folder, every):
        self.api, self.repo, self.folder, self.every = api, repo, Path(folder), every

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.every == 0:
            model.save_pretrained(str(self.folder))
            tokenizer.save_pretrained(str(self.folder))
            self.api.upload_folder(repo_id=self.repo, folder_path=str(self.folder),
                                   commit_message=f"{RUN_NAME} step {state.global_step} checkpoint, "
                                                  f"code {GITHUB}/commit/{CODE_COMMIT}")
            print(f"step {state.global_step}: adapter uploaded to {self.repo}")

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

if SKIP_TRAINING:
    print("SKIP_TRAINING: no trainer")
else:
    api.create_repo(GRPO_ADAPTER_REPO, private=True, exist_ok=True)
    train_ds = Dataset.from_list([{"prompt": p, "meta": r["meta"]} for p, r in zip(prompts, grpo_train)])
    trainer = GRPOTrainer(model=model, processing_class=text_tokenizer, reward_funcs=[gym_reward],
                          train_dataset=train_ds, args=GRPOConfig(
        output_dir=str(RUN_DIR / "trainer"), seed=SEED, report_to="none", save_strategy="no", logging_steps=1,
        num_generations=G, per_device_train_batch_size=G,
        gradient_accumulation_steps=PROMPTS_PER_STEP,  # TRL generates G x this answers at once: 2 prompts -> left padding
        temperature=TEMPERATURE, max_completion_length=MAX_COMPLETION, beta=BETA,
        learning_rate=LEARNING_RATE, lr_scheduler_type="constant_with_warmup", warmup_steps=WARMUP_STEPS,
        optim="adamw_8bit", adam_beta1=0.9, adam_beta2=0.99, weight_decay=0.1, max_grad_norm=0.1,
        max_steps=MAX_STEPS, bf16=BF16, fp16=not BF16,
    ), callbacks=[TimeBudget(TIME_BUDGET_MIN), ValCheck(VAL_EVERY),
                  HubCheckpoint(api, GRPO_ADAPTER_REPO, RUN_DIR / "checkpoint", UPLOAD_EVERY)])
    a = trainer.args
    print(f"{a.per_device_train_batch_size} answers x {a.gradient_accumulation_steps} accumulation = {PROMPTS_PER_STEP} "
          f"prompts x {a.num_generations} answers per step; temperature {a.temperature}, beta {a.beta}, "
          f"lr {a.learning_rate} ({a.lr_scheduler_type}), max {a.max_completion_length} new tokens, {a.max_steps} steps")
    # Unsloth silently changes the batch size if it isn't a multiple of num_generations
    assert (a.per_device_train_batch_size, a.gradient_accumulation_steps, a.num_generations) == (G, PROMPTS_PER_STEP, G)
    assert (a.temperature, a.beta, a.max_completion_length) == (TEMPERATURE, BETA, MAX_COMPLETION)
    lora_dtypes = {str(p.dtype) for p in model.parameters() if p.requires_grad}
    base_dtypes = {str(p.dtype) for p in model.parameters() if not p.requires_grad}
    print(f"trained (LoRA) weights: {lora_dtypes}, frozen base weights: {base_dtypes}")  # the real precision
    print(f"flags passed: bf16={a.bf16}, fp16={a.fp16}")

## 9. Train (or load the finished run)

Up to 300 steps or the time budget. As soon as training ends, before anything else can fail, the final adapter and the
logs are uploaded with "final" in the commit message: everything after this point can be redone with `SKIP_TRAINING`.
With `SKIP_TRAINING = True` this section only downloads `log_history.json` and `val_checks.json` from `START_ADAPTER`.

**Check:** the steps, time per step, total time and peak memory, then "final adapter and logs uploaded" (or "SKIP_TRAINING: loaded …").

In [ ]:
import shutil

ADAPTER_DIR = RUN_DIR / "adapter"
LOGS = ("log_history.json", "val_checks.json")
if SKIP_TRAINING:
    for name in LOGS:
        shutil.copy(api.hf_hub_download(repo_id=START_ADAPTER, filename=name), RUN_DIR / name)
    saved = json.loads((RUN_DIR / "val_checks.json").read_text())
    VAL_HISTORY, PADDING_CHECK = saved["checks"], saved["padding_check"]
    BATCHED_GENERATION, PROMPTS_PER_STEP = PADDING_CHECK["passed"], PADDING_CHECK["prompts_per_step"]
    print(f"SKIP_TRAINING: loaded {', '.join(LOGS)} from {START_ADAPTER}; batched generation: {BATCHED_GENERATION}")
else:
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    trainer.train()
    minutes, steps = (time.time() - start) / 60, trainer.state.global_step
    print(f"{steps} steps in {minutes:.1f} min ({60 * minutes / steps:.1f} s per step), "
          f"peak GPU memory {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")
    (RUN_DIR / "log_history.json").write_text(json.dumps(trainer.state.log_history, indent=1))
    model.save_pretrained(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    for name in LOGS:
        shutil.copy(RUN_DIR / name, ADAPTER_DIR / name)
    api.upload_folder(repo_id=GRPO_ADAPTER_REPO, folder_path=str(ADAPTER_DIR),
                      commit_message=f"{RUN_NAME} final adapter after {steps} steps, code {GITHUB}/commit/{CODE_COMMIT}")
    print(f"final adapter and logs uploaded to https://huggingface.co/{GRPO_ADAPTER_REPO}")

history = json.loads((RUN_DIR / "log_history.json").read_text())
assert history and VAL_HISTORY and VAL_HISTORY[0]["step"] == 0, "log_history.json or the step-0 val check is missing"

## 10. Curves

One point per training step (16 answers, so noisy: the thick line is a 10-step moving average), plus the val check every
50 steps on its 40 fixed rows. What to look for: reward and all_correct going up, duplicates and false refusals going down,
refusals going up, `frac_reward_zero_std` not climbing toward 1 (no signal left) and the answer length not drifting.

**Check:** six panels, saved as `grpo_curves.png`.

In [ ]:
import matplotlib.pyplot as plt


def series(key):
    pts = [(e["step"], e[key]) for e in history if e.get(key) is not None]
    assert pts, f"{key} is missing from log_history.json"
    return zip(*pts)


def moving_average(ys, k=10):
    return [sum(ys[max(0, i - k + 1):i + 1]) / len(ys[max(0, i - k + 1):i + 1]) for i in range(len(ys))]


PANELS = [  # (title, [(log key, label)], val-check key or None)
    ("reward", [("reward", "train")], "reward"),
    ("all_correct", [("gym/all_correct", "train")], "all_correct"),
    ("duplicate rate", [("gym/duplicate_rate", "train")], None),
    ("refusals", [("gym/refusal_rate", "refusal (impossible)"), ("gym/false_refusal_rate", "false refusal (feasible)")], None),
    ("frac_reward_zero_std", [("frac_reward_zero_std", "train")], None),
    ("mean answer length (tokens)", [("completions/mean_length", "train")], None),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (title, lines, val_key) in zip(axes.flat, PANELS):
    for (key, label), color in zip(lines, ["#2a78d6", "#eb6834"]):
        xs, ys = series(key)
        ax.plot(xs, ys, color=color, alpha=0.25, linewidth=1)
        ax.plot(xs, moving_average(ys), color=color, linewidth=2, label=label)
    if val_key:
        ax.plot([h["step"] for h in VAL_HISTORY], [h["means"][val_key] for h in VAL_HISTORY],
                color="#1b9e47", marker="o", markersize=6, linewidth=2, label="val check (40 rows)")
    ax.set(title=title, xlabel="step")
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8)
fig.suptitle(f"{RUN_NAME}: GRPO on top of the SFT adapter")
fig.tight_layout()
fig.savefig(RUN_DIR / "grpo_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Final val check

The same 40 val rows, greedy, now with the trained model, next to the step-0 check (the starting model). Still val, not
test: the real comparison with the baselines and SFT is `evaluate.py` on the Mac (section 15). The answers are saved for the
reload test.

**Check:** one table; all_correct and refusal_rate above step 0, duplicate_rate below.

In [ ]:
final = val_check(model, "final")
(RUN_DIR / "val_check_final.json").write_text(json.dumps(final, indent=1))
VAL_TABLE = means_table({"step 0": VAL_HISTORY[0]["means"], "final": final["means"]})
print(VAL_TABLE)
assert len(final["answers"]) == len(VAL_SET) and final["means"]["parse_ok"] is not None
for row, a in list(zip(VAL_SET, final["answers"]))[:3]:
    print("\nUSER:  ", row["messages"][1]["content"], "\nMODEL: ", a["output"])

## 12. Save the adapter and the merged model

As in SFT: the adapter is small, and the merged model (LoRA added into the base) is what the Mac evaluates. Any tensor the
merge wrote in another dtype is cast back to the base's dtype, so the GRPO result doesn't mix training with a precision
change, and the saved tokenizer must still build exactly the `evaluate.py` prompt.

**Check:** shards only BF16 (plus a few F32), torch_dtype bfloat16, "tokenizer check passed".

In [ ]:
MERGED_DIR = RUN_DIR / "merged"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")

In [ ]:
from huggingface_hub import snapshot_download
from safetensors import safe_open
from safetensors.torch import load_file, save_file

TORCH_DTYPE = {"BF16": torch.bfloat16, "F16": torch.float16, "F32": torch.float32}


def header_dtypes(path):
    with safe_open(str(path), "pt") as f:
        return {k: f.get_slice(k).get_dtype() for k in f.keys()}


# the base file is named model.safetensors-00001-of-00001.safetensors, so read every base shard, whatever its name
base_dir = Path(snapshot_download(MODEL_NAME, allow_patterns=["*.safetensors"]))  # in the HF cache since section 4
base = {}
for f in sorted(base_dir.glob("*.safetensors")):
    base.update(header_dtypes(f))
assert base, f"no .safetensors files found for {MODEL_NAME}"
print(f"base: {len(base)} tensors from {[f.name for f in base_dir.glob('*.safetensors')]}")

for shard in sorted(MERGED_DIR.glob("*.safetensors")):
    saved = header_dtypes(shard)
    wrong = {k for k, d in saved.items() if d in TORCH_DTYPE and d != base.get(k, "BF16")}
    if wrong:
        tensors = load_file(str(shard))
        for k in wrong:
            tensors[k] = tensors[k].to(TORCH_DTYPE[base.get(k, "BF16")])
        save_file(tensors, str(shard), metadata={"format": "pt"})
    print(f"{shard.name}: {dict(collections.Counter(header_dtypes(shard).values()))}, cast {len(wrong)} tensors, "
          f"{len(saved.keys() & base.keys())}/{len(saved)} names found in the base")

config_path = MERGED_DIR / "config.json"
config = json.loads(config_path.read_text())
for c in (config, config.get("text_config", {})):
    for key in ("torch_dtype", "dtype"):
        if key in c:
            c[key] = "bfloat16"
config_path.write_text(json.dumps(config, indent=2))

In [ ]:
for folder in (ADAPTER_DIR, MERGED_DIR):
    print(folder)
    for f in sorted(folder.iterdir()):
        print(f"  {f.name:40} {f.stat().st_size / 1e6:10.1f} MB")
    saved_tok = AutoTokenizer.from_pretrained(str(folder))
    assert all(build_prompt(saved_tok, r["messages"][:2]) == build_prompt(reference, r["messages"][:2]) for r in val), folder

config = json.loads((MERGED_DIR / "config.json").read_text())
text_config = config.get("text_config", {})
print("architectures:", config.get("architectures"))
print("torch_dtype:  ", config.get("torch_dtype", config.get("dtype")),
      "| text_config:", text_config.get("torch_dtype", text_config.get("dtype")))
print("tokenizer check passed: the saved chat templates give the same build_prompt output")

## 13. Reload test

We load the merged model back, as the Mac will, and answer the first 8 val rows again: the same batch as in section 11,
so with greedy decoding they should match. If an answer stops parsing or loses more than 0.1 reward, the run stops here,
before the broken model is uploaded. The training model is freed first, so the two don't share the GPU.

**Check:** 8 × MATCH, and no assert error. A DIFF with about the same reward is a tiny rounding difference (merged bf16 vs
base + LoRA).

In [ ]:
import gc

for name in ("trainer", "model"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

reloaded, _ = FastLanguageModel.from_pretrained(str(MERGED_DIR), max_seq_length=MAX_SEQ_LENGTH,
                                                load_in_4bit=False, load_in_16bit=True, fast_inference=False)
FastLanguageModel.for_inference(reloaded)
for row, (new, _), old in zip(VAL_SET, greedy_answers(reloaded, VAL_SET[:VAL_BATCH]), final["answers"]):
    check = verify(row["meta"], new)
    print("MATCH" if new == old["output"] else "DIFF ", f"reward {check['reward']:.2f} (section 11: {old['reward']:.2f})",
          repr(new[:70]))
    assert check["parse_ok"] >= old["parse_ok"] and check["reward"] >= old["reward"] - 0.1, \
        "the reloaded merged model answers worse: not uploading it"

## 14. Upload to the Hugging Face Hub

As in SFT: `upload_folder` uploads exactly the files we checked, each upload is a versioned commit that names the code
commit, and the model cards make a portfolio page. Both repos are private. The merged repo also gets the log, the curves
and the val checks; the adapter repo already has the log and the step-by-step checkpoints in its history.

**Check:** two private repo URLs printed; each model page shows its card.

In [ ]:
steps_trained = max(e["step"] for e in history)


def model_card(kind):
    repo = GRPO_ADAPTER_REPO if kind == "adapter" else GRPO_MERGED_REPO
    if kind == "merged":
        usage = f"```bash\npython src/evaluate.py generate --run {RUN_NAME} --model {GRPO_MERGED_REPO}\n```"
        chart = f"![GRPO curves](https://huggingface.co/{GRPO_MERGED_REPO}/resolve/main/grpo_curves.png)\n\n"
    else:
        usage = f"Load it with PEFT on top of `{MODEL_NAME}`, or use the merged model: [{GRPO_MERGED_REPO}](https://huggingface.co/{GRPO_MERGED_REPO})."
        chart = ""
    return f"""---
base_model: {MODEL_NAME}
library_name: {"peft" if kind == "adapter" else "transformers"}
tags: [grpo, rl, lora, unsloth, trl, json, fitness]
---

# {repo}

{MODEL_NAME} trained in two stages to write gym workouts as JSON from requests like "chest workout, medium difficulty,
free weights": SFT ([{SFT_ADAPTER}](https://huggingface.co/{SFT_ADAPTER})), then **GRPO** (this model), which continues
training the SFT LoRA adapter (start: `{START_ADAPTER}`). It answers `{{"workout": [{{"name", "sets", "reps"}}, ...]}}` with
exercise names from the [exercise catalog](https://github.com/hasaneyldrm/exercises-dataset), or `{{"error": ...}}` when the
request is impossible. This repo holds the **{"LoRA adapter" if kind == "adapter" else "merged 16-bit model (base + LoRA, bf16)"}**.

- Code: [{GITHUB}]({GITHUB}), commit [`{CODE_COMMIT[:7]}`]({GITHUB}/commit/{CODE_COMMIT})
- Data: tag [`data-v1`]({GITHUB}/tree/data-v1/data/processed); GRPO prompts: {steps_trained * PROMPTS_PER_STEP} of {N_PROMPTS} train rows drawn ({PROMPT_MIX["impossible"]:.0%} impossible requests); the test set was not used here

## Reward

The reward is the project's verifier, `verify(request, answer)["reward"]` from `src/verifier.py`, in [0, 1]: 10% for the
exact JSON format, 90% for content (names in the catalog 0.30, body part 0.15, equipment 0.15, count, no duplicates,
sets/reps and no advanced exercise in easy/medium 0.10 each). An impossible request scores 1 only for an error answer.
Names that differ only by "v. N", "(male)" or "(female)" are the same exercise.

## Training

| setting | value |
|---|---|
| method | GRPO with TRL `GRPOTrainer` + Unsloth, continuing the SFT LoRA (r=16, alpha=16) |
| answers per prompt / prompts per step | {G} / {PROMPTS_PER_STEP} |
| sampling temperature / max new tokens | {TEMPERATURE} / {MAX_COMPLETION} |
| KL penalty (beta) | {BETA} |
| learning rate | {LEARNING_RATE}, constant after {WARMUP_STEPS} warmup steps; 8-bit AdamW (0.9, 0.99), weight decay 0.1, max grad norm 0.1 |
| steps | {steps_trained} (max {MAX_STEPS}, time budget {TIME_BUDGET_MIN} min) |
| libraries / seed | trl {trl.__version__}, unsloth {unsloth.__version__}; other GRPO settings at TRL's defaults / {SEED} |
| GPU | {torch.cuda.get_device_name(0)} |

{chart}## Val check

{len(VAL_SET)} val rows (13 impossible + 27 feasible), greedy decoding, scored with `verify()`. Step 0 is the starting
model. The real evaluation, on the 300-row test set, runs separately on a Mac with MLX.

{VAL_TABLE}

## Known limit

**Little or no refusal signal on some impossible requests.** Before training, the SFT model answered 8 impossible train
requests 8 times each at temperature 0.7, the training temperature. **chest / bands** never refused (0 of 8): no refusal for
GRPO to reward, so refusals on this pair may not improve. **shoulders / bodyweight** refused 1 of 8, a weak signal.
forearms / bodyweight refused 0 of 8 on one request (easy, 3 exercises) but 2 of 8 on the other (hard, 6).

## Usage

{usage}
"""


for name in LOGS:
    shutil.copy(RUN_DIR / name, ADAPTER_DIR / name)
for name in (*LOGS, "grpo_curves.png", "val_check_final.json"):
    shutil.copy(RUN_DIR / name, MERGED_DIR / name)
for kind, repo, folder in [("adapter", GRPO_ADAPTER_REPO, ADAPTER_DIR), ("merged", GRPO_MERGED_REPO, MERGED_DIR)]:
    api.create_repo(repo, private=True, exist_ok=True)
    (folder / "README.md").write_text(model_card(kind))  # replaces the auto-generated card
    api.upload_folder(repo_id=repo, folder_path=str(folder),
                      commit_message=f"{RUN_NAME} final {kind}, trained with {GITHUB}/commit/{CODE_COMMIT}")
    print(f"{kind:8} -> https://huggingface.co/{repo}")
print(f"\nOn the Mac: python src/evaluate.py generate --run {RUN_NAME} --model {GRPO_MERGED_REPO}")

## 15. Next steps (on the Mac)

The repos are private, so the Mac reads the token from `.env` (not committed). Replace `<HF_USER>` with yours;
`mlx_lm.load()` downloads the model into `HF_HOME` the first time.
```bash
conda activate GymLeaderProject
cd "/Volumes/lexar/produttività/projects/SFT-Gym-Workout-Program"
set -a; source .env; set +a    # puts HF_TOKEN in this terminal's environment
caffeinate -i python src/evaluate.py generate --run sft_grpo --model <HF_USER>/qwen3.5-2b-gym-grpo
python src/evaluate.py score --run sft_grpo
python src/analyze_errors.py --run sft_grpo
# optional: how much learning signal is left after GRPO
HF_HUB_OFFLINE=1 python src/probe_groups.py --run grpo_t1 --model <HF_USER>/qwen3.5-2b-gym-grpo
```

**Check:** `eval/results/results.md` gets an `sft_grpo` row, next to the baselines and `sft_lora_r16`.